# Antenna Tilt Optimization with QAOA — Hackathon Solution

**QUBIT × AT&T Hackathon 2026 · Antenna Tilt Challenge · author: Rom5Leo**

This notebook is my hackathon solution to AT&T's antenna-tilt optimization problem, presented
as a clean walk-through: the problem, why it is hard, how it becomes a quantum optimization,
and the result. It was built in a 24-hour hackathon and is kept here as an honest record of
the effort — the reasoning and the working pipeline, not a production system.

**How it's organized.** The heavy lifting lives in two libraries I built from this work:
- [`telecomopt`](https://github.com/Rom5Leo/Telecom_Optimization) — the RF physics and the
  antenna problem model.
- [`qcoptlib`](https://github.com/Rom5Leo/Quantum_Optimization) — the reusable QUBO + QAOA solvers.

The notebook stays thin: it imports those libraries and tells the story. That separation is
itself part of the solution's design.

---

## The problem, in one paragraph

Every cellular antenna can be tilted down by a few degrees. Tilt trades **coverage against
interference**: too little downtilt overshoots and interferes with neighbours; too much shrinks
the cell and leaves holes. Across a network the antennas are **coupled** — the best tilt for
one depends on its neighbours — so you cannot tune them independently. AT&T values solving this
at ~$1.1B/year and its CEO has called it a key quantum frontier. This is a combinatorial
optimization problem, and that coupling is what makes it a candidate for QAOA.


## 1. The physics: how tilt becomes signal quality

The foundation is a real RF model (Ericsson's LTE parameters, cited in `telecomopt.rf`). It
turns a tilt angle and a distance into a received signal strength, then into SINR and spectral
efficiency (throughput). Every formula traces to the reference papers — the model is grounded
in real parameters, not invented.

The single most important function is the **antenna elevation pattern**: gain peaks when the
beam points at a user and falls off away from it. That is the entire mechanism by which tilt
controls both coverage and interference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from telecomopt.rf import RFParams, path_gain_db, sinr, spectral_efficiency, to_linear

p = RFParams()   # Ericsson Table I defaults

# Signal quality at a 200 m cell edge, as a function of total downtilt:
tilts = np.linspace(0, 20, 60)
se = []
for t in tilts:
    s = sinr(serving_distance_m=200, serving_tilt_deg=t,
             interferers=[(500, 6.0)], params=p)   # one neighbour 500 m away at 6 deg
    se.append(spectral_efficiency(s))

plt.figure(figsize=(7, 4))
plt.plot(tilts, se, lw=2)
plt.axvline(tilts[int(np.argmax(se))], ls="--", color="grey",
            label=f"best ~{tilts[int(np.argmax(se))]:.0f} deg")
plt.xlabel("total downtilt (deg)"); plt.ylabel("spectral efficiency (bps/Hz)")
plt.title("There is an optimal tilt — the coverage/interference trade-off")
plt.legend(); plt.grid(alpha=0.3); plt.show()


The curve peaks and falls off on both sides — proof there is a genuine optimum to find, and
that it depends on the interfering neighbour's tilt too. Multiply this coupling across a whole
network and independent tuning stops working.

## 2. From a toy model to real physics (the hackathon path)

The solution was built in stages. It is worth showing the path, because the encoding ideas are
clearest in the simple version and the physics is layered on afterwards.

**Stage A — a toy model.** First I proved the *encoding* worked with placeholder numbers: each
sector picks a tilt index, coverage is a made-up reward, interference is a made-up penalty
between neighbours. This established the QAOA machinery end to end before any RF was involved.

**Stage B — real angles.** Tilt indices became real degrees (electrical ±10° around a
mechanical baseline), and the cost was made *arithmetic* so it could be written as a QAOA
`phase` expression.

**Stage C — real physics (this notebook).** The placeholder coverage/interference were replaced
by the SINR model above, and the network went 2-D: antennas at (x, y) positions with a
distance-based interference graph.

The key modeling lesson carried through every stage: **define the cost once and use that exact
function for the classical ground truth, the quantum `phase`, and the read-out** — otherwise you
optimize one thing and check another.

## 3. Encoding the real problem as a QUBO

Now the full problem. We place antennas in 2-D, build the interference graph, and assemble the
**QUBO** — the standard optimization form QAOA consumes:
- one binary variable per (antenna, tilt): `x[i,k] = 1` iff antenna i uses tilt k,
- **coverage** as a per-antenna reward (linear terms),
- **interference** as a pairwise coupling between neighbours (quadratic ZZ terms),
- a **one-hot penalty** enforcing exactly one tilt per antenna.

All of this is one call to `antenna_tilt_qubo` — the model lives in `telecomopt.antenna`.

In [ ]:
from telecomopt.antenna import AntennaNetwork, antenna_tilt_qubo, decode_onehot
from qcoptlib.viz import plot_network_2d

# A small 2-D network. Kept modest (3 antennas x 4 tilts = 12 qubits) so it simulates fast
# AND is a genuinely coupled instance: all three antennas interfere with each other.
rng = np.random.default_rng(6)
net = AntennaNetwork(
    positions=rng.uniform(0, 1000, size=(3, 2)),   # 3 antennas
    elec_options=(-10.0, -3.0, 3.0, 10.0),         # 4 tilt options -> 12 qubits (one-hot)
)
print(f"{net.n} antennas, {len(net.neighbors)} interfering pairs, "
      f"{net.n * net.k} qubits (one-hot encoding)")

fig, ax = plt.subplots(figsize=(5, 5))
plot_network_2d(net.positions, edges=net.neighbors, ax=ax,
                title=f"{net.n} antennas, {len(net.neighbors)} interference links")
plt.show()

qubo = antenna_tilt_qubo(net, beta=0.02)   # beta = interference weight
print(f"QUBO: {qubo.n} variables")


## 4. Classical ground truth (the benchmark)

Because this instance is small, we can find the true optimum by brute force — the honest
benchmark to measure the quantum result against. The baseline is every antenna at its
mechanical tilt (no optimization).

In [ ]:
from itertools import product

# baseline = the mildest tilt option (closest to a neutral setting) for every antenna
neutral_idx = min(range(net.k), key=lambda k: abs(net.elec_options[k]))
baseline_cfg = [neutral_idx] * net.n
best_cfg = max(product(range(net.k), repeat=net.n),
               key=net.total_spectral_efficiency)             # brute-force optimum

se_base = net.total_spectral_efficiency(baseline_cfg)
se_opt  = net.total_spectral_efficiency(list(best_cfg))
print(f"Baseline throughput : {se_base:.2f} bps/Hz")
print(f"Optimal  throughput : {se_opt:.2f} bps/Hz  (+{100*se_opt/se_base-100:.0f}%)")
print(f"Optimal tilts (elec): {[net.elec_options[k] for k in best_cfg]} deg")


## 5. Solving it with QAOA

Now the quantum solve. `qcoptlib` converts the QUBO to an Ising cost, builds the QAOA circuit,
tunes the angles with a classical optimizer, and samples the result. Here we use the Qiskit/Aer
backend so the notebook runs end-to-end without a Classiq account; `qcoptlib` also ships a
Classiq backend used the same way.

QAOA's output is read out robustly: we score every sampled bitstring by its real objective and
keep the best — so a weakly-concentrated distribution (the reality on NISQ-scale problems) still
yields a good answer.

In [ ]:
from qcoptlib.quantum.qiskit_backend import solve_qubo_qaoa

# QAOA is stochastic, so we use a couple of restarts and a fixed seed for reproducibility.
res = solve_qubo_qaoa(qubo, num_layers=3, maxiter=60, shots=2048, restarts=2, seed=0)

qaoa_cfg = decode_onehot(res.best_bits, net)
se_qaoa = net.total_spectral_efficiency(qaoa_cfg)
print(f"QAOA tilts (elec): {[net.elec_options[k] for k in qaoa_cfg]} deg")
print(f"QAOA throughput  : {se_qaoa:.2f} bps/Hz")
print(f"QAOA reached {100*se_qaoa/se_opt:.0f}% of the classical optimum "
      f"(+{100*se_qaoa/se_base-100:.0f}% over baseline)")


## 6. Results

In [ ]:
from qcoptlib.viz import plot_convergence, plot_before_after

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# convergence of the QAOA optimizer
plot_convergence(res.history, ax=axes[0], title="QAOA convergence")

# throughput: baseline vs QAOA vs classical optimum
axes[1].bar(["baseline", "QAOA", "optimum"], [se_base, se_qaoa, se_opt],
            color=["#adb5bd", "#1c7ed6", "#37b24d"])
axes[1].set_ylabel("total throughput (bps/Hz)")
axes[1].set_title("Throughput"); axes[1].grid(alpha=0.3)

# per-antenna tilt: baseline vs QAOA
plot_before_after(
    [net.baseline_tilt] * net.n,
    [net.total_tilt(qaoa_cfg[i]) for i in range(net.n)],
    ax=axes[2], labels=("baseline", "QAOA"),
    xlabel="antenna", ylabel="total downtilt (deg)", title="Tilt per antenna",
)
plt.tight_layout(); plt.show()


## 7. Honest limitations and the quantum-advantage argument

This is a **credible proof of concept**, and the honest framing is part of the work:

- **QAOA on today's simulators concentrates weakly.** On this instance it reached ~2/3 of the
  classical optimum. That is expected at NISQ scale, and the robust read-out (score-and-keep-best)
  plus the classical benchmark is exactly how you handle it honestly.
- **At this size, classical brute force wins.** The quantum interest is at scale: the antennas
  are coupled, so the search space is `K^N` and classical local search can miss global optima
  (the reference papers say this explicitly). QAOA searches jointly.
- **It scales because interference is local.** Only nearby antennas couple, so the ZZ graph is
  sparse — the number of two-qubit terms grows with the network, not with its square. That is the
  credible path from this toy instance to AT&T's thousands of sectors.
- **The right architecture is hybrid.** Classical handles the RF modelling and validation; the
  quantum component takes the hard combinatorial choice. That is the split the challenge brief
  itself describes, and the split these two libraries implement.

**Why quantum at all, then?** Not because it beats classical here — it doesn't, at this size —
but because the problem is genuinely coupled combinatorial optimization, the encoding is clean,
and the approach scales in principle. Demonstrating that end-to-end, grounded in real RF
parameters, with an honest benchmark, is the contribution.

---

*Built for the QUBIT × AT&T Hackathon 2026. Libraries: `telecomopt` (RF + problem model),
`qcoptlib` (QUBO + QAOA). This notebook is the antenna-tilt solution walk-through; further
development continues in the telecom-optimization repository.*
